# Diff-Pruning — DDPM CIFAR-10 (clean pipeline)

This notebook runs the baseline pruning pipeline end-to-end on Google Colab.

**Important:** the runtime `diffusers/` patches that used to live in code cells are now baked into the repo files
(`diffusers/__init__.py`, `diffusers/pipelines/__init__.py`, `diffusers/models/attention_processor.py`).
For this notebook to work, **those changes must be committed and pushed to the GitHub fork that the next cell clones.**
If you cloned before pushing, the patch cells would be needed again.

This test run uses a **light pruning ratio (0.1)** and a **short fine-tune (3000 steps)** so the whole pipeline
finishes inside one Colab session. For a faithful baseline result, use ratio `0.3` and `num_iters=100000`.

In [ ]:
!git clone https://github.com/elliotcanter11/Diff-Pruning.git

In [ ]:
%cd Diff-Pruning/

In [ ]:
!pip install -r requirements.txt

## Data & pretrained model

In [ ]:
# Download and extract CIFAR-10 images to data/cifar10_images for training and evaluation.
!python tools/extract_cifar10_hug.py --output data

In [ ]:
# Download an official DDPM model and convert it to the Huggingface Diffusers format.
# The converted (EMA) model of google/ddpm-cifar10-32 lands in pretrained/ddpm_ema_cifar10.
!bash tools/convert_cifar10_ddpm_ema.sh

## Prune

Creates a pruned model at `run/pruned/ddpm_cifar10_pruned`. The argument is the pruning ratio
(0.1 = light, for a fast test; use 0.3 to reproduce the paper's baseline).

In [ ]:
!bash scripts/prune_ddpm_cifar10.sh 0.1  # pruning ratio = 10%

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("run/pruned/ddpm_cifar10_pruned/vis/after_pruning.png")

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis("off")
plt.show()

## Fine-tune

Recovers quality lost to pruning and saves to `run/finetuned/ddpm_cifar10_pruned_post_training`.
The argument is the number of iterations (`num_iters`). 3000 is a quick sanity run; the baseline uses 100000.

Checkpoints are written every `save_model_steps` (1000), so Colab persistence / resuming across a dropped
session is possible from the last saved checkpoint.

In [ ]:
!bash scripts/finetune_ddpm_cifar10.sh 3000  # num_iters = 3000 (quick test)

In [ ]:
import glob
import re
from PIL import Image
import matplotlib.pyplot as plt

vis_dir = "run/finetuned/ddpm_cifar10_pruned_post_training/vis"
paths = glob.glob(f"{vis_dir}/iter-*.png")

# sort by the step number embedded in the filename
paths.sort(key=lambda p: int(re.search(r"iter-(\d+)\.png", p).group(1)))

fig, axes = plt.subplots(1, len(paths), figsize=(5 * len(paths), 5))
if len(paths) == 1:
    axes = [axes]

for ax, path in zip(axes, paths):
    step = re.search(r"iter-(\d+)\.png", path).group(1)
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f"step {step}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Sample

In [ ]:
# Pruned + finetuned: sample and save images to run/sample/ddpm_cifar10_pruned
!bash scripts/sample_ddpm_cifar10_pruned.sh

In [ ]:
# Pretrained baseline: sample and save images to run/sample/ddpm_cifar10_pretrained
!bash scripts/sample_ddpm_cifar10_pretrained.sh

## Evaluate (FID)

FID is only meaningful with a large number of generated samples (the paper uses 50k). A short test run
with few samples will give a poor / noisy FID — treat it as a smoke test that the metric runs, not a result.

In [ ]:
# Pre-compute the stats of the CIFAR-10 dataset
!python fid_score.py --save-stats data/cifar10_images run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256

In [ ]:
# Compute the FID score of the sampled images
!python fid_score.py run/sample/ddpm_cifar10_pruned run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256